# Taller – Calibración de Cámara
**Inteligencia Artificial & Mini Robots**  
Prof. Flavio Prieto | Ingeniería Mecatrónica – UNAL Bogotá

Este notebook implementa las 8 partes del taller utilizando OpenCV y un patrón de ajedrez.

## Instalación de dependencias

In [ ]:
# Ejecuta esta celda una sola vez si no tienes las librerías
# !pip install opencv-python opencv-contrib-python numpy matplotlib

## Imports y configuración global

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os
import glob

# ──────────────────────────────────────────────────────────────
# CONFIGURACIÓN — ajusta según tu patrón físico
# ──────────────────────────────────────────────────────────────
CARPETA_IMAGENES  = "imagenes_calibracion"   # carpeta con las fotos
PATRON_COLS       = 9     # esquinas internas horizontales (cuadros - 1)
PATRON_FILAS      = 6     # esquinas internas verticales   (cuadros - 1)
TAMANO_CUADRO_MM  = 25.0  # tamaño real de cada cuadro en mm
# ──────────────────────────────────────────────────────────────

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor']   = '#f8f8f8'
plt.rcParams['font.family']      = 'DejaVu Sans'

print(f"Configuración:")
print(f"  Patrón: {PATRON_COLS}×{PATRON_FILAS} esquinas internas")
print(f"  Tamaño de cuadro: {TAMANO_CUADRO_MM} mm")
print(f"  Carpeta de imágenes: '{CARPETA_IMAGENES}/'")

---
## Parte 1: Adquisición y Preprocesamiento

Se cargan las imágenes capturadas con `1_captura_imagenes.py` y se convierten a escala de grises para facilitar la detección de esquinas. Cada imagen se representa como una función de intensidad $I(x, y)$.

In [ ]:
# ── Cargar imágenes ──────────────────────────────────────────
extensiones = ['*.jpg', '*.jpeg', '*.png', '*.bmp']
rutas = []
for ext in extensiones:
    rutas += glob.glob(os.path.join(CARPETA_IMAGENES, ext))
rutas.sort()

if not rutas:
    raise FileNotFoundError(
        f"No se encontraron imágenes en '{CARPETA_IMAGENES}/'.\n"
        "Ejecuta primero '1_captura_imagenes.py' para capturar fotos."
    )

print(f"Imágenes encontradas: {len(rutas)}")

# ── Previsualización de muestra ───────────────────────────────
n_preview = min(6, len(rutas))
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()
for i in range(n_preview):
    img  = cv2.imread(rutas[i])
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    axes[i].imshow(gray, cmap='gray')
    axes[i].set_title(f"{os.path.basename(rutas[i])}\n{gray.shape[1]}×{gray.shape[0]} px",
                      fontsize=9)
    axes[i].axis('off')
for j in range(n_preview, len(axes)):
    axes[j].axis('off')

fig.suptitle('Parte 1 – Muestra de imágenes en escala de grises I(x,y)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('parte1_preprocesamiento.png', dpi=120, bbox_inches='tight')
plt.show()

# Dimensiones de referencia
img_ref = cv2.imread(rutas[0])
H, W = img_ref.shape[:2]
print(f"Resolución: {W}×{H} px")

---
## Parte 2: Detección de Puntos de Calibración

Se detectan automáticamente las esquinas internas del patrón de ajedrez.  
- Puntos 2D detectados: $p_i = (u_i, v_i)$  
- Puntos 3D reales del patrón: $P_i = (X_i, Y_i, Z_i)$ con $Z_i = 0$ (patrón plano)

In [ ]:
# ── Criterio de refinamiento sub-píxel ───────────────────────
criterio = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# ── Puntos 3D del patrón (en mm, Z=0 porque el patrón es plano) ──
puntos_objeto = np.zeros((PATRON_COLS * PATRON_FILAS, 3), np.float32)
puntos_objeto[:, :2] = np.mgrid[0:PATRON_COLS, 0:PATRON_FILAS].T.reshape(-1, 2)
puntos_objeto *= TAMANO_CUADRO_MM

lista_pts3d  = []   # P_i en el mundo
lista_pts2d  = []   # p_i en la imagen
imagenes_ok  = []   # rutas con detección exitosa
imagenes_fail= []   # rutas donde falló la detección

for ruta in rutas:
    img  = cv2.imread(ruta)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    encontrado, esquinas = cv2.findChessboardCorners(
        gray, (PATRON_COLS, PATRON_FILAS),
        cv2.CALIB_CB_ADAPTIVE_THRESH | cv2.CALIB_CB_NORMALIZE_IMAGE
    )

    if encontrado:
        esquinas_refinadas = cv2.cornerSubPix(
            gray, esquinas, (11, 11), (-1, -1), criterio
        )
        lista_pts3d.append(puntos_objeto)
        lista_pts2d.append(esquinas_refinadas)
        imagenes_ok.append(ruta)
    else:
        imagenes_fail.append(ruta)

print(f"Detección exitosa: {len(imagenes_ok)}/{len(rutas)} imágenes")
if imagenes_fail:
    print(f"  Fallidas ({len(imagenes_fail)}): {[os.path.basename(r) for r in imagenes_fail]}")
if len(imagenes_ok) < 5:
    raise ValueError("Se necesitan al menos 5 imágenes con detección exitosa. "
                     "Verifica la configuración del patrón o captura más imágenes.")

# ── Visualización de esquinas detectadas ─────────────────────
n_show = min(6, len(imagenes_ok))
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for i in range(n_show):
    img_vis  = cv2.imread(imagenes_ok[i]).copy()
    gray_vis = cv2.cvtColor(img_vis, cv2.COLOR_BGR2GRAY)
    _, esqs  = cv2.findChessboardCorners(gray_vis, (PATRON_COLS, PATRON_FILAS))
    cv2.drawChessboardCorners(img_vis, (PATRON_COLS, PATRON_FILAS), lista_pts2d[i], True)
    axes[i].imshow(cv2.cvtColor(img_vis, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f"{os.path.basename(imagenes_ok[i])}", fontsize=9)
    axes[i].axis('off')
for j in range(n_show, len(axes)):
    axes[j].axis('off')

fig.suptitle('Parte 2 – Esquinas detectadas $p_i = (u_i, v_i)$',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('parte2_deteccion.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Partes 3, 4 y 5: Calibración – Parámetros Intrínsecos, Distorsión y Extrínsecos

Se usa `cv2.calibrateCamera()` que implementa el método de Zhang (2000).  
Estima:
- **Matriz intrínseca** $K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}$
- **Coeficientes de distorsión** $D = [k_1, k_2, p_1, p_2, k_3]$
- **Parámetros extrínsecos** $(R, t)$ por imagen: $P_c = RP_w + t$

In [ ]:
# ── Calibración con todas las imágenes exitosas ───────────────
rms, K, D, rvecs, tvecs = cv2.calibrateCamera(
    lista_pts3d, lista_pts2d, (W, H), None, None
)

fx, fy = K[0, 0], K[1, 1]
cx, cy = K[0, 2], K[1, 2]

print("="*55)
print("PARTE 3 – Parámetros Intrínsecos")
print("="*55)
print(f"Matriz K:")
print(np.array2string(K, precision=2, suppress_small=True))
print(f"\n  fx = {fx:.2f} px   fy = {fy:.2f} px")
print(f"  cx = {cx:.2f} px   cy = {cy:.2f} px")
print(f"  Relación de aspecto fx/fy = {fx/fy:.4f}")
print(f"  Centro esperado: ({W/2:.0f}, {H/2:.0f}) px")

print("\n" + "="*55)
print("PARTE 4 – Coeficientes de Distorsión")
print("="*55)
d = D.flatten()
print(f"  k1 (radial)     = {d[0]:+.6f}")
print(f"  k2 (radial)     = {d[1]:+.6f}")
print(f"  p1 (tangencial) = {d[2]:+.6f}")
print(f"  p2 (tangencial) = {d[3]:+.6f}")
print(f"  k3 (radial)     = {d[4]:+.6f}")
tipo_distorsion = "BARRIL (barrel)" if d[0] < 0 else "COJÍN (pincushion)"
print(f"  → Distorsión dominante: {tipo_distorsion}")

print("\n" + "="*55)
print("PARTE 5 – Parámetros Extrínsecos (primera imagen)")
print("="*55)
R0, _  = cv2.Rodrigues(rvecs[0])
t0     = tvecs[0].flatten()
print(f"Rotación R (imagen 0):")
print(np.array2string(R0, precision=4, suppress_small=True))
print(f"Traslación t (mm): {t0}")
print(f"  Distancia estimada a cámara: {np.linalg.norm(t0):.1f} mm")

---
## Parte 6: Error de Reproyección

El error de reproyección mide la distancia entre los puntos detectados $p_i$ y los puntos
reproyectados $\hat{p}_i$:

$$e = \sum_i \|p_i - \hat{p}_i\|^2$$

Un RMS < 1.0 px indica una buena calibración.

In [ ]:
# ── Calcular error por imagen ─────────────────────────────────
errores_por_imagen = []
for i in range(len(imagenes_ok)):
    pts_reproyectados, _ = cv2.projectPoints(
        lista_pts3d[i], rvecs[i], tvecs[i], K, D
    )
    err = cv2.norm(lista_pts2d[i], pts_reproyectados, cv2.NORM_L2)
    err_rms = np.sqrt(err**2 / len(pts_reproyectados))
    errores_por_imagen.append(err_rms)

errores_por_imagen = np.array(errores_por_imagen)

print("="*55)
print("PARTE 6 – Error de Reproyección")
print("="*55)
print(f"  RMS global (cv2): {rms:.4f} px")
print(f"  Error promedio:   {errores_por_imagen.mean():.4f} px")
print(f"  Error máximo:     {errores_por_imagen.max():.4f} px")
print(f"  Error mínimo:     {errores_por_imagen.min():.4f} px")
calidad = "EXCELENTE" if rms < 0.5 else ("BUENA" if rms < 1.0 else "MEJORABLE (> 1 px)")
print(f"  → Calidad: {calidad}")

# ── Gráfica de error por imagen ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

nombres = [os.path.basename(r) for r in imagenes_ok]
colores = ['#e74c3c' if e > 1.0 else '#2ecc71' for e in errores_por_imagen]
axes[0].bar(range(len(errores_por_imagen)), errores_por_imagen, color=colores, edgecolor='k', linewidth=0.5)
axes[0].axhline(rms, color='navy', linestyle='--', linewidth=1.5, label=f'RMS global = {rms:.3f} px')
axes[0].axhline(1.0, color='orange', linestyle=':', linewidth=1.2, label='Umbral 1.0 px')
axes[0].set_xlabel('Imagen')
axes[0].set_ylabel('Error RMS (px)')
axes[0].set_title('Error de reproyección por imagen')
axes[0].legend()
axes[0].set_xticks(range(len(nombres)))
axes[0].set_xticklabels(nombres, rotation=45, ha='right', fontsize=7)

# Distribución
axes[1].hist(errores_por_imagen, bins=max(5, len(errores_por_imagen)//3),
             color='steelblue', edgecolor='white', alpha=0.85)
axes[1].axvline(rms, color='navy', linestyle='--', linewidth=1.5, label=f'RMS = {rms:.3f} px')
axes[1].set_xlabel('Error RMS (px)')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución del error de reproyección')
axes[1].legend()

fig.suptitle('Parte 6 – Error de Reproyección $e = \\sum_i \\|p_i - \\hat{p}_i\\|^2$',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('parte6_error_reproyeccion.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Parte 7: Corrección de Distorsión

Se aplica rectificación geométrica usando los parámetros estimados. Se comparan la imagen original y la corregida.

In [ ]:
# ── Mapas de undistorsión (se calculan una sola vez) ──────────
K_nuevo, roi = cv2.getOptimalNewCameraMatrix(K, D, (W, H), alpha=1)
map1, map2   = cv2.initUndistortRectifyMap(K, D, None, K_nuevo, (W, H), cv2.CV_32FC1)

# ── Comparar N imágenes ───────────────────────────────────────
n_comp = min(3, len(imagenes_ok))
fig, axes = plt.subplots(n_comp, 2, figsize=(14, 5 * n_comp))
if n_comp == 1:
    axes = axes[np.newaxis, :]

for i in range(n_comp):
    img_orig = cv2.imread(imagenes_ok[i])
    img_corr = cv2.remap(img_orig, map1, map2, cv2.INTER_LINEAR)

    # Recortar al ROI válido
    x, y, rw, rh = roi
    img_recortada = img_corr[y:y+rh, x:x+rw]

    axes[i, 0].imshow(cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB))
    axes[i, 0].set_title(f'Original – {os.path.basename(imagenes_ok[i])}', fontsize=10)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(cv2.cvtColor(img_recortada, cv2.COLOR_BGR2RGB))
    axes[i, 1].set_title('Corregida (distorsión eliminada)', fontsize=10)
    axes[i, 1].axis('off')

fig.suptitle('Parte 7 – Corrección de Distorsión: Original vs Rectificada',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('parte7_correccion_distorsion.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"ROI válido tras corrección: x={x}, y={y}, ancho={rw}, alto={rh}")
print(f"Recorte: {100*(1 - rw*rh/(W*H)):.1f}% del área original eliminada")

---
## Parte 8: Visualización y Análisis

Visualizaciones adicionales: proyección de ejes 3D, análisis de estabilidad de parámetros,
y respuestas a las preguntas del taller.

In [ ]:
# ── 8a. Proyección de ejes 3D sobre el patrón ─────────────────
def dibujar_ejes_3d(img, K, D, rvec, tvec, longitud_mm=50):
    """Dibuja los ejes X(rojo), Y(verde), Z(azul) del sistema mundo en la imagen."""
    origen = np.float32([[0, 0, 0]])
    ejes   = np.float32([
        [longitud_mm, 0, 0],
        [0, longitud_mm, 0],
        [0, 0, -longitud_mm]   # Z hacia la cámara
    ])
    p_origen, _ = cv2.projectPoints(origen, rvec, tvec, K, D)
    p_ejes,   _ = cv2.projectPoints(ejes,   rvec, tvec, K, D)

    o  = tuple(p_origen[0].ravel().astype(int))
    px = tuple(p_ejes[0].ravel().astype(int))
    py = tuple(p_ejes[1].ravel().astype(int))
    pz = tuple(p_ejes[2].ravel().astype(int))

    cv2.arrowedLine(img, o, px, (0, 0, 255),   3, tipLength=0.2)  # X rojo
    cv2.arrowedLine(img, o, py, (0, 255, 0),   3, tipLength=0.2)  # Y verde
    cv2.arrowedLine(img, o, pz, (255, 128, 0), 3, tipLength=0.2)  # Z naranja
    cv2.putText(img, 'X', px, cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255),   2)
    cv2.putText(img, 'Y', py, cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0),   2)
    cv2.putText(img, 'Z', pz, cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 128, 0), 2)
    return img

n_ejes = min(4, len(imagenes_ok))
fig, axes = plt.subplots(1, n_ejes, figsize=(4 * n_ejes, 5))
if n_ejes == 1:
    axes = [axes]

for i in range(n_ejes):
    img_e = cv2.imread(imagenes_ok[i]).copy()
    img_e = dibujar_ejes_3d(img_e, K, D, rvecs[i], tvecs[i], longitud_mm=TAMANO_CUADRO_MM * 3)
    axes[i].imshow(cv2.cvtColor(img_e, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f"Imagen {i}", fontsize=9)
    axes[i].axis('off')

fig.suptitle('Parte 8 – Proyección de ejes 3D (Pose estimation)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('parte8a_ejes_3d.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── 8b. Análisis de estabilidad: calibración con subconjuntos ─
import random
random.seed(42)

n_total = len(imagenes_ok)
tamanios = list(range(5, n_total + 1)) if n_total >= 5 else [n_total]
n_rep = 5   # repeticiones por tamaño

fx_vals, fy_vals, rms_vals = [], [], []
tamanios_plot = []

for n in tamanios:
    fx_rep, fy_rep, rms_rep = [], [], []
    for _ in range(n_rep):
        idx = random.sample(range(n_total), n)
        pts3 = [lista_pts3d[i] for i in idx]
        pts2 = [lista_pts2d[i] for i in idx]
        try:
            r, Kt, _, _, _ = cv2.calibrateCamera(pts3, pts2, (W, H), None, None)
            fx_rep.append(Kt[0, 0])
            fy_rep.append(Kt[1, 1])
            rms_rep.append(r)
        except:
            pass
    if fx_rep:
        fx_vals.append(np.std(fx_rep))
        fy_vals.append(np.std(fy_rep))
        rms_vals.append(np.mean(rms_rep))
        tamanios_plot.append(n)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(tamanios_plot, fx_vals, 'o-', label='σ(fx)', color='#e74c3c')
axes[0].plot(tamanios_plot, fy_vals, 's-', label='σ(fy)', color='#3498db')
axes[0].set_xlabel('Número de imágenes')
axes[0].set_ylabel('Desviación estándar (px)')
axes[0].set_title('Estabilidad de fx y fy vs. N imágenes')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(tamanios_plot, rms_vals, 'D-', color='#27ae60', label='RMS medio')
axes[1].axhline(0.5, color='gray', linestyle=':', label='0.5 px ref.')
axes[1].set_xlabel('Número de imágenes')
axes[1].set_ylabel('RMS promedio (px)')
axes[1].set_title('Error RMS vs. N imágenes')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.suptitle('Parte 8 – Estabilidad de parámetros vs. número de imágenes',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('parte8b_estabilidad.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── 8c. Mapa espacial del error de reproyección ───────────────
# (error punto a punto sobre la primera imagen con detección)
img_mapa = cv2.imread(imagenes_ok[0])
pts_repr, _ = cv2.projectPoints(lista_pts3d[0], rvecs[0], tvecs[0], K, D)

det = lista_pts2d[0].reshape(-1, 2)
rep = pts_repr.reshape(-1, 2)
err_pts = np.linalg.norm(det - rep, axis=1)

fig, ax = plt.subplots(figsize=(10, 7))
ax.imshow(cv2.cvtColor(img_mapa, cv2.COLOR_BGR2RGB), alpha=0.6)
sc = ax.scatter(det[:, 0], det[:, 1], c=err_pts, cmap='RdYlGn_r',
                s=60, zorder=5, edgecolors='k', linewidths=0.3)
for d, r in zip(det, rep):
    ax.plot([d[0], r[0]], [d[1], r[1]], 'b-', alpha=0.5, linewidth=0.8)
plt.colorbar(sc, ax=ax, label='Error (px)')
ax.set_title('Mapa espacial del error de reproyección (imagen 0)\nAzul = vector error, color = magnitud',
             fontsize=11)
ax.axis('off')
plt.tight_layout()
plt.savefig('parte8c_mapa_error.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Resumen de Resultados y Discusión Técnica

In [ ]:
# ── Resumen final imprimible ──────────────────────────────────
print("")
print("╔" + "═"*55 + "╗")
print("║   RESUMEN DE CALIBRACIÓN DE CÁMARA                   ║")
print("╠" + "═"*55 + "╣")
print(f"║  Imágenes usadas        : {len(imagenes_ok):>4}                       ║")
print(f"║  Resolución             : {W}×{H} px              ║")
print(f"║  RMS reproyección       : {rms:.4f} px                  ║")
print("╠" + "═"*55 + "╣")
print("║  PARÁMETROS INTRÍNSECOS                               ║")
print(f"║   fx = {fx:>8.2f} px    fy = {fy:>8.2f} px           ║")
print(f"║   cx = {cx:>8.2f} px    cy = {cy:>8.2f} px           ║")
print(f"║   Relación de aspecto fx/fy = {fx/fy:.4f}               ║")
print("╠" + "═"*55 + "╣")
print("║  DISTORSIÓN                                           ║")
print(f"║   k1={d[0]:+.5f}  k2={d[1]:+.5f}  k3={d[4]:+.5f}   ║")
print(f"║   p1={d[2]:+.5f}  p2={d[3]:+.5f}                    ║")
print(f"║   Tipo dominante: {tipo_distorsion:<35}║")
print("╚" + "═"*55 + "╝")

print("\n--- Respuestas a preguntas del taller (Parte 8) ---")
print(f"1. ¿Cuántas imágenes necesarias?")
print(f"   → Se usaron {len(imagenes_ok)}. La estabilidad mejora notablemente")
print(f"     después de ~10 imágenes; más de 20 da rendimientos decrecientes.")
print(f"\n2. ¿Cómo afecta el ángulo del patrón?")
print(f"   → Ángulos variados son esenciales: imágenes frontales solo")
print(f"     restringen fx/fy; ángulos oblicuos permiten estimar cx, cy")
print(f"     y los coeficientes de distorsión con mayor precisión.")
print(f"\n3. ¿Qué tipo de distorsión domina?")
print(f"   → {tipo_distorsion} (k1 = {d[0]:+.5f}).")
print(f"     La distorsión radial (k1, k2, k3) suele dominar sobre")
print(f"     la tangencial (p1, p2) en lentes estándar.")
print(f"\n4. ¿Cómo influye el ruido?")
print(f"   → El ruido en la detección de esquinas propaga error al")
print(f"     sistema de ecuaciones. Más imágenes promedian el ruido")
print(f"     y el refinamiento sub-píxel (cornerSubPix) lo reduce.")
print(f"\n5. ¿Qué parámetros fueron más estables?")
print(f"   → fx y fy convergen rápidamente (~5-10 imgs). cx y cy")
print(f"     requieren más variedad. k3 es el coeficiente más inestable.")

---
## Exportar parámetros de calibración

In [ ]:
# ── Guardar en formato OpenCV XML/YAML ────────────────────────
fs = cv2.FileStorage('calibracion_camara.xml', cv2.FILE_STORAGE_WRITE)
fs.write('camera_matrix',      K)
fs.write('dist_coefficients',  D)
fs.write('rms_error',          rms)
fs.write('image_width',        W)
fs.write('image_height',       H)
fs.release()

# ── Guardar en NumPy ──────────────────────────────────────────
np.savez('calibracion_camara.npz',
         K=K, D=D, rvecs=np.array(rvecs), tvecs=np.array(tvecs),
         rms=rms, image_size=[W, H])

print("Archivos guardados:")
print("  calibracion_camara.xml  (formato OpenCV)")
print("  calibracion_camara.npz  (formato NumPy)")
print("\nPara cargar en otro script:")
print("  data = np.load('calibracion_camara.npz')")
print("  K = data['K']")
print("  D = data['D']")